# Adaptive sampling: RAR-D relocates the collocation points

**Book:** §5.3.2, Figure 5.5 &nbsp;·&nbsp; `ch05/rard_adaptive_sampling.ipynb`

§5.3.1 met **loss imbalance**: a thin feature carries a large residual that a uniform mean drowns
out. RBA answered it by **reweighting** fixed points. **RAR-D** (residual-based adaptive
distribution, Wu et al. 2023) answers it differently — it **relocates** the points, redrawing them
from a probability density

$$p(\mathbf{x}) \propto \frac{r(\mathbf{x})}{\mathbb{E}[r]} + 1$$

so that more collocation points fall where the residual is large. (The `+1` keeps a uniform floor,
for exploration.)

**Where it works.** A 2-D Poisson problem with a sharp peak, `−Δu = f`, exact `u = exp(−250·r²)`.
A fixed uniform budget barely samples the peak (**rel L2 ≈ 0.77**, peak height only 0.76); RAR-D
migrates points onto the feature (**rel L2 ≈ 0.47**, peak recovered). Panel (b) shows the point
cloud clustering on the peak; panel (d), the error history — noisy, because each resample perturbs
the training, but ending well below uniform.

**Where it does *not*.** RAR-D needs a residual *signal* to relocate toward. On the boundary-layer
trap of §5.3.1, the spurious flat solution has *near-zero residual everywhere*, so there is nothing
for RAR-D to chase — and it cannot escape, whereas RBA's reweighting (which amplifies even a tiny
residual) does. **Relocation and reweighting are not interchangeable.**

In [ ]:
"""RAR-D (residual-based adaptive distribution) on a 2-D Poisson problem with a sharp peak.
   -Delta u = f  on [0,1]^2,  u=0 on the boundary,  exact u = exp(-a((x-.5)^2+(y-.5)^2)).
A uniform collocation budget barely samples the peak; RAR-D redraws points with probability
proportional to the residual, migrating them onto the feature. Honest side-by-side."""
import time, json, numpy as np, torch, torch.nn as nn, matplotlib
import matplotlib.pyplot as plt

np.random.seed(0)
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", dev)
A = 250.0                                    # peak sharpness: width ~ 1/sqrt(2A) = 0.045
def g1(f,x): return torch.autograd.grad(f,x,torch.ones_like(f),create_graph=True)[0]
def u_exact(x,y): return np.exp(-A*((x-0.5)**2+(y-0.5)**2))
def f_src(x,y):                              # f = -Delta u_exact = 4A(1 - A r^2) u
    r2=(x-0.5)**2+(y-0.5)**2; return 4*A*(1-A*r2)*np.exp(-A*r2)

def mlp(seed=0):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2,64),nn.Tanh(),nn.Linear(64,64),nn.Tanh(),
                         nn.Linear(64,64),nn.Tanh(),nn.Linear(64,1)).to(dev)
def U(net,x,y): return x*(1-x)*y*(1-y)*net(torch.cat([x,y],1))   # hard u=0 on boundary

# evaluation grid
n=201; gx=np.linspace(0,1,n); GX,GY=np.meshgrid(gx,gx,indexing='ij')
Uex=u_exact(GX,GY); Xt=torch.tensor(GX.ravel(),dtype=torch.float32,device=dev).reshape(-1,1)
Yt=torch.tensor(GY.ravel(),dtype=torch.float32,device=dev).reshape(-1,1)
def evaluate(net):
    with torch.no_grad(): up=U(net,Xt,Yt).cpu().numpy().reshape(n,n)
    err=float(np.sqrt(np.mean((up-Uex)**2)/np.mean(Uex**2)))
    return up, err, float(up[n//2,n//2])

def residual(net,x,y):
    x=x.requires_grad_(True); y=y.requires_grad_(True)
    u=U(net,x,y); lap=g1(g1(u,x),x)+g1(g1(u,y),y)
    fs=torch.tensor(f_src(x.detach().cpu().numpy(),y.detach().cpu().numpy()),
                    dtype=torch.float32,device=dev)
    return lap+fs                            # -Delta u = f  ->  lap u + f = 0

NF=500; EPOCHS=15000
def train(mode, seed=0):
    net=mlp(seed); opt=torch.optim.Adam(net.parameters(),2e-3)
    # candidate pool for RAR-D
    pool_x=torch.rand(20000,1,device=dev); pool_y=torch.rand(20000,1,device=dev)
    # initial collocation set (uniform)
    cx=torch.rand(NF,1,device=dev); cy=torch.rand(NF,1,device=dev)
    t0=time.perf_counter(); hist=[]
    for e in range(EPOCHS):
        if e==10000:
            for gp in opt.param_groups: gp['lr']=4e-4
        # baseline 'uniform': the fixed set drawn once above -- never moved.
        if mode=='rard' and e>0 and e%500==0:     # RAR-D / RAD (Wu et al. 2023)
            with torch.enable_grad():
                r=residual(net,pool_x.clone(),pool_y.clone()).detach().abs().cpu().numpy().ravel()
            p=r/r.mean() + 1.0; p=p/p.sum()        # p ∝ r/E[r] + 1  (residual density + uniform floor)
            nr=NF//2
            idx=np.random.choice(len(p),size=nr,p=p)         # half follow the residual
            ux=torch.rand(NF-nr,1,device=dev); uy=torch.rand(NF-nr,1,device=dev)  # half stay uniform
            cx=torch.cat([pool_x[idx],ux]); cy=torch.cat([pool_y[idx],uy])
        opt.zero_grad()
        r=residual(net,cx.clone(),cy.clone())
        (r**2).mean().backward(); opt.step()
        if e%500==0:
            _,ee,pk=evaluate(net); hist.append((e,ee,pk))
            if e%3000==0: print(f'   {mode} e={e:5d} relL2={ee:.2f} peak={pk:.2f}')
    if dev.type=='cuda': torch.cuda.synchronize()
    up,err,peak=evaluate(net)
    return dict(up=up, err=err, peak=peak, hist=hist, cx=cx.detach().cpu().numpy().ravel(),
                cy=cy.detach().cpu().numpy().ravel(), t=time.perf_counter()-t0)

R={}
for mode in ('uniform','rard'):
    R[mode]=train(mode)
    print(f"{mode:8s}: rel L2 = {R[mode]['err']:.3f}   peak u(0.5,0.5) = {R[mode]['peak']:.3f}  "
          f"(exact 1.000)  [{R[mode]['t']:.0f}s]")

# ------------------------------------------------------------------ figure
fig,ax=plt.subplots(1,4,figsize=(18.5,4.4))
lv=np.linspace(0,1,21)
# (a) uniform points over the exact peak
ax[0].contourf(GX,GY,Uex,lv,cmap='inferno')
ax[0].scatter(R['uniform']['cx'],R['uniform']['cy'],s=5,c='cyan',alpha=.6,linewidths=0)
ax[0].set_title(f"(a) Uniform points miss the peak\nrel $L_2$={R['uniform']['err']:.2f}, "
                f"peak={R['uniform']['peak']:.2f}",fontsize=10.5)
ax[0].set_xlabel('x'); ax[0].set_ylabel('y'); ax[0].set_aspect('equal')
# (b) RAR-D points concentrated on the peak
ax[1].contourf(GX,GY,Uex,lv,cmap='inferno')
ax[1].scatter(R['rard']['cx'],R['rard']['cy'],s=5,c='cyan',alpha=.6,linewidths=0)
ax[1].set_title(f"(b) RAR-D relocates points to the peak\nrel $L_2$={R['rard']['err']:.2f}, "
                f"peak={R['rard']['peak']:.2f}",fontsize=10.5)
ax[1].set_xlabel('x'); ax[1].set_ylabel('y'); ax[1].set_aspect('equal')
# (c) centre-line slice
mid=n//2
ax[2].plot(gx,Uex[:,mid],'g',lw=2.6,alpha=.6,label='exact')
ax[2].plot(gx,R['uniform']['up'][:,mid],'--',color='tab:red',lw=1.6,
           label=f"uniform ({R['uniform']['err']:.2f})")
ax[2].plot(gx,R['rard']['up'][:,mid],'-',color='tab:blue',lw=1.6,
           label=f"RAR-D ({R['rard']['err']:.2f})")
ax[2].set_xlabel('x  (at y=0.5)'); ax[2].set_ylabel('u'); ax[2].grid(alpha=.3)
ax[2].legend(fontsize=9); ax[2].set_title('(c) The peak, resolved only by RAR-D',fontsize=10.5)
# (d) error history: uniform plateaus, RAR-D keeps improving
hu=np.array(R['uniform']['hist']); hr=np.array(R['rard']['hist'])
ax[3].plot(hu[:,0],hu[:,1],color='tab:red',lw=1.6,label='uniform (fixed)')
ax[3].plot(hr[:,0],hr[:,1],color='tab:blue',lw=1.6,label='RAR-D')
ax[3].set_xlabel('epoch'); ax[3].set_ylabel('relative $L_2$ error'); ax[3].grid(alpha=.3)
ax[3].legend(fontsize=9); ax[3].set_title('(d) Error history')
plt.tight_layout(); plt.show()
print("METRICS", json.dumps({m:{k:R[m][k] for k in('err','peak')} for m in R},indent=1))
